In [1]:
!pip install torchscale

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 14.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 41.9 MB/s eta 0:00:00
  Created wheel for fairscale: filename=fairscale-0.4.0-py3-none-any.whl size=240019 sha256=9f4fd5ed40a813b98dba255179614296e59cae81e15579a47c363c4c50a9d537
  Stored in directory: /root/.cache/pip/wheels/cd/10/4a/b6cf78bb3d0689c08b5f75d3d5ce39055ce86588580b04104e
Successfully built fairscale
  Attempting uninstall: timm
    Found existing installation: timm 1.0.27
    Uninstalling timm-1.0.27:
      Successfully uninstalled timm-1.0.27


In [2]:
import torchscale
print("TorchScale installed successfully")

TorchScale installed successfully


In [3]:
! pip install transformers

In [4]:
import torchscale.architecture.retnet as retnet
from torchscale.architecture.retnet import RetNetDecoder
from torchscale.architecture.retnet import RetNetRelPos

print(dir(retnet))

['DecoderLayer', 'DropPath', 'F', 'GLU', 'MOELayer', 'MultiScaleRetention', 'RMSNorm', 'RetNetDecoder', 'RetNetRelPos', 'Top1Gate', 'Top2Gate', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'checkpoint_wrapper', 'init_bert_params', 'make_experts', 'math', 'nn', 'np', 'torch', 'wrap']


In [5]:
! pip install rwkv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.4/411.4 kB 28.3 MB/s eta 0:00:00


In [6]:
! pip install pynvml

In [7]:
# ── Core Python ──────────────────────────────────────────────
import os
import math
import time
import json
import warnings
warnings.filterwarnings('ignore')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── PyTorch Core ─────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# ── HuggingFace ──────────────────────────────────────────────
from datasets import load_dataset
from transformers import AutoTokenizer

# ── RetNet (TorchScale) ──────────────────────────────────────
from torchscale.architecture.config import RetNetConfig
from torchscale.architecture.retnet import RetNetDecoder

# ── RWKV ─────────────────────────────────────────────────────
from rwkv.model import RWKV
from rwkv.utils import PIPELINE

# ── GPU Monitoring ───────────────────────────────────────────
import pynvml
try:
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    GPU_AVAILABLE = True
    print("✅ GPU monitoring initialized")
except Exception as e:
    GPU_AVAILABLE = False
    print(f"⚠️ pynvml not available: {e}")

# ── Data & Visualization ─────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# ── Device Setup ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
      if torch.cuda.is_available() else "VRAM: N/A")

✅ GPU monitoring initialized
Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB


In [8]:
# ── Global Config ─────────────────────────────────────────────
VOCAB_SIZE   = 50257       # GPT-2 tokenizer
DROPOUT      = 0.1
BATCH_SIZE   = 8
EPOCHS       = 5
LR           = 3e-4
SEQ_LENGTHS  = [128]
# In Cell 3 Config — reduce model size
EMBED_DIM  = 128    # was 256
NUM_LAYERS = 3      # was 4
NUM_HEADS  = 4      # keep same

print("✅ Config ready")
print(f"   Vocab size  : {VOCAB_SIZE}")
print(f"   Embed dim   : {EMBED_DIM}")
print(f"   Num heads   : {NUM_HEADS}")
print(f"   Num layers  : {NUM_LAYERS}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   Epochs      : {EPOCHS}")
print(f"   LR          : {LR}")
print(f"   Seq lengths : {SEQ_LENGTHS}")

✅ Config ready
   Vocab size  : 50257
   Embed dim   : 128
   Num heads   : 4
   Num layers  : 3
   Batch size  : 8
   Epochs      : 5
   LR          : 0.0003
   Seq lengths : [128]


In [10]:
# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded")
print(f"   Vocab size : {tokenizer.vocab_size}")
print(f"   Pad token  : {tokenizer.pad_token}")
print(f"   EOS token  : {tokenizer.eos_token}")

✅ Tokenizer loaded
   Vocab size : 50257
   Pad token  : <|endoftext|>
   EOS token  : <|endoftext|>


In [12]:
# ── Dataset Class ─────────────────────────────────────────────
class TinyStoriesDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        self.examples = []
        self.seq_len  = seq_len

        print(f"  Tokenizing {len(data)} stories at seq_len={seq_len}...")

        for item in data:
            tokens = tokenizer(
                item['text'],
                max_length=seq_len + 1,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
            ids = tokens['input_ids'].squeeze(0)
            self.examples.append(ids)

        print(f"  ✅ Done — {len(self.examples)} examples ready")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        tokens = self.examples[idx]
        x = tokens[:-1]    # input
        y = tokens[1:]     # target shifted by 1
        return x, y

In [13]:
# ── Load Raw Data ─────────────────────────────────────────────
print("Loading TinyStories...")
train_data = load_dataset("roneneldan/TinyStories", split="train[:9000]")
val_data   = load_dataset("roneneldan/TinyStories", split="validation[:1000]")

print(f"✅ Train stories : {len(train_data)}")
print(f"✅ Val stories   : {len(val_data)}")

Loading TinyStories...


✅ Train stories : 9000
✅ Val stories   : 1000


In [14]:
# ── Build Loaders for All 3 Sequence Lengths ──────────────────
dataloaders = {}

for seq_len in SEQ_LENGTHS:
    print(f"\nBuilding loaders for seq_len={seq_len}...")

    train_dataset = TinyStoriesDataset(train_data, tokenizer, seq_len)
    val_dataset   = TinyStoriesDataset(val_data,   tokenizer, seq_len)

    dataloaders[seq_len] = {
        'train' : DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True),
        'val'   : DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    }

    print(f"  ✅ seq_len={seq_len} → "
          f"train: {len(dataloaders[seq_len]['train'])} batches | "
          f"val: {len(dataloaders[seq_len]['val'])} batches")

print("\n✅ All dataloaders ready!")

# ── Verify Shapes ─────────────────────────────────────────────
print("\nVerifying shapes...")
for seq_len in SEQ_LENGTHS:
    x, y = next(iter(dataloaders[seq_len]['train']))
    print(f"  seq_len={seq_len} → x: {x.shape} | y: {y.shape}")


Building loaders for seq_len=128...
  Tokenizing 9000 stories at seq_len=128...
  ✅ Done — 9000 examples ready
  Tokenizing 1000 stories at seq_len=128...
  ✅ Done — 1000 examples ready
  ✅ seq_len=128 → train: 1125 batches | val: 125 batches

✅ All dataloaders ready!

Verifying shapes...
  seq_len=128 → x: torch.Size([8, 128]) | y: torch.Size([8, 128])


In [17]:
# ── Transformer Model ─────────────────────────────────────────
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads,
                 num_layers, seq_len, dropout=0.1):
        super().__init__()

        self.seq_len   = seq_len
        self.embed_dim = embed_dim

        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb   = nn.Embedding(seq_len, embed_dim)
        self.dropout   = nn.Dropout(dropout)

        # Transformer Decoder Layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        # Output Head
        self.fc_out = nn.Linear(embed_dim, vocab_size)

        # Initialize Weights
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T = x.shape

        # Positions
        positions = torch.arange(T, device=x.device).unsqueeze(0)

        # Embeddings
        x = self.dropout(
            self.token_emb(x) + self.pos_emb(positions)
        )

        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(
            T, device=x.device
        )

        # Forward
        x = self.transformer(x, x, tgt_mask=mask)

        # Logits
        return self.fc_out(x)


# ── Parameter Counter ─────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Initialize Transformer for All Seq Lengths ────────────────
transformer_models = {}

for seq_len in SEQ_LENGTHS:
    model = TransformerModel(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        seq_len=seq_len,
        dropout=DROPOUT
    ).to(device)

    transformer_models[seq_len] = model
    print(f"✅ Transformer seq_len={seq_len} → "
          f"Parameters: {count_params(model):,}")


# ── Sanity Check Forward Pass ─────────────────────────────────
print("\nRunning forward pass check...")
model = transformer_models[128]
model.eval()

x, y = next(iter(dataloaders[128]['train']))
x = x.to(device)

with torch.no_grad():
    output = model(x)

print(f"  Input  : {x.shape}")
print(f"  Output : {output.shape}")
print(f"✅ Transformer forward pass works!")

✅ Transformer seq_len=128 → Parameters: 13,726,161

Running forward pass check...
  Input  : torch.Size([8, 128])
  Output : torch.Size([8, 128, 50257])
✅ Transformer forward pass works!


In [19]:
# ── RWKV Model (from scratch) ─────────────────────────────────

class RWKVTimeMixing(nn.Module):
    """
    Core RWKV mechanism — replaces self attention
    Uses weighted key value with time decay
    """
    def __init__(self, embed_dim, seq_len):
        super().__init__()

        self.embed_dim = embed_dim

        # Time decay and bonus parameters
        self.time_decay  = nn.Parameter(torch.randn(embed_dim))
        self.time_bonus  = nn.Parameter(torch.randn(embed_dim))

        # Token shift parameters
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, embed_dim) * 0.5)
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, embed_dim) * 0.5)
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, embed_dim) * 0.5)

        # Projections
        self.key        = nn.Linear(embed_dim, embed_dim, bias=False)
        self.value      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.receptance = nn.Linear(embed_dim, embed_dim, bias=False)
        self.output     = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        # Token shift — mix current and previous token
        shifted = torch.cat([torch.zeros(B, 1, C, device=x.device), x[:, :-1]], dim=1)

        # Mix
        xk = x * self.time_mix_k + shifted * (1 - self.time_mix_k)
        xv = x * self.time_mix_v + shifted * (1 - self.time_mix_v)
        xr = x * self.time_mix_r + shifted * (1 - self.time_mix_r)

        # Compute K V R
        k = self.key(xk)
        v = self.value(xv)
        r = torch.sigmoid(self.receptance(xr))

        # Simplified time weighted attention
        decay  = torch.exp(-torch.exp(self.time_decay)).unsqueeze(0).unsqueeze(0)
        bonus  = self.time_bonus.unsqueeze(0).unsqueeze(0)

        # Causal weighted sum
        weights  = torch.exp(k + bonus) * decay
        weighted = weights * v
        out      = r * weighted

        return self.output(out)


class RWKVChannelMixing(nn.Module):
    """
    RWKV FFN replacement — channel mixing
    """
    def __init__(self, embed_dim):
        super().__init__()

        self.time_mix_k = nn.Parameter(torch.ones(1, 1, embed_dim) * 0.5)
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, embed_dim) * 0.5)

        # Projections
        self.key        = nn.Linear(embed_dim, embed_dim * 4, bias=False)
        self.value      = nn.Linear(embed_dim * 4, embed_dim, bias=False)
        self.receptance = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        # Token shift
        shifted = torch.cat([torch.zeros(B, 1, C, device=x.device), x[:, :-1]], dim=1)

        # Mix
        xk = x * self.time_mix_k + shifted * (1 - self.time_mix_k)
        xr = x * self.time_mix_r + shifted * (1 - self.time_mix_r)

        # Channel mix
        k   = torch.square(torch.relu(self.key(xk)))
        v   = self.value(k)
        r   = torch.sigmoid(self.receptance(xr))

        return r * v


class RWKVBlock(nn.Module):
    """
    Single RWKV block
    Time Mixing + Channel Mixing with residual connections
    """
    def __init__(self, embed_dim, seq_len):
        super().__init__()

        self.ln1          = nn.LayerNorm(embed_dim)
        self.ln2          = nn.LayerNorm(embed_dim)
        self.time_mixing  = RWKVTimeMixing(embed_dim, seq_len)
        self.channel_mixing = RWKVChannelMixing(embed_dim)

    def forward(self, x):
        # Time mixing with residual
        x = x + self.time_mixing(self.ln1(x))
        # Channel mixing with residual
        x = x + self.channel_mixing(self.ln2(x))
        return x


class RWKVModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_layers,
                 seq_len, dropout=0.1):
        super().__init__()

        self.embed_dim = embed_dim
        self.seq_len   = seq_len

        # Embedding
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.dropout   = nn.Dropout(dropout)

        # RWKV Blocks
        self.blocks = nn.ModuleList([
            RWKVBlock(embed_dim, seq_len)
            for _ in range(num_layers)
        ])

        # Output
        self.ln_out  = nn.LayerNorm(embed_dim)
        self.fc_out  = nn.Linear(embed_dim, vocab_size)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        # Embedding
        x = self.dropout(self.token_emb(x))

        # RWKV blocks
        for block in self.blocks:
            x = block(x)

        # Output
        x = self.ln_out(x)
        return self.fc_out(x)


# ── Initialize RWKV for All Seq Lengths ───────────────────────
rwkv_models = {}

for seq_len in SEQ_LENGTHS:
    model = RWKVModel(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        num_layers=NUM_LAYERS,
        seq_len=seq_len,
        dropout=DROPOUT
    ).to(device)

    rwkv_models[seq_len] = model
    print(f"✅ RWKV seq_len={seq_len} → "
          f"Parameters: {count_params(model):,}")


# ── Sanity Check ──────────────────────────────────────────────
print("\nRunning RWKV forward pass check...")
model = rwkv_models[128]
model.eval()

x, y = next(iter(dataloaders[128]['train']))
x    = x.to(device)

with torch.no_grad():
    output = model(x)

print(f"  Input  : {x.shape}")
print(f"  Output : {output.shape}")
print(f"✅ RWKV forward pass works!")

✅ RWKV seq_len=128 → Parameters: 13,559,505

Running RWKV forward pass check...
  Input  : torch.Size([8, 128])
  Output : torch.Size([8, 128, 50257])
✅ RWKV forward pass works!


In [21]:
print("\nParameter Count Comparison:")
print(f"  Transformer : {count_params(transformer_models[128]):,}")
print(f"  RWKV        : {count_params(rwkv_models[128]):,}")


Parameter Count Comparison:
  Transformer : 13,726,161
  RWKV        : 13,559,505


In [23]:
# ── RetNet Model (TorchScale) ─────────────────────────────────
from torchscale.architecture.config import RetNetConfig
from torchscale.architecture.retnet import RetNetDecoder

class RetNetModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads,
                 num_layers, seq_len, dropout=0.1):
        super().__init__()

        self.embed_dim = embed_dim
        self.seq_len   = seq_len

        # ── Token Embedding ───────────────────────────────────
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.dropout   = nn.Dropout(dropout)

        # ── RetNet Config ─────────────────────────────────────
        config = RetNetConfig(
            decoder_embed_dim=embed_dim,
            decoder_retention_heads=num_heads,
            decoder_ffn_embed_dim=embed_dim * 4,
            decoder_layers=num_layers,
            dropout=dropout,
            activation_dropout=dropout,
            no_scale_embedding=True,
            layernorm_embedding=False,
            vocab_size=vocab_size,
        )

        # ── RetNet Decoder ────────────────────────────────────
        self.retnet = RetNetDecoder(
            config,
            embed_tokens=self.token_emb
        )

        # ── Output Head ───────────────────────────────────────
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape

        # RetNet expects (seq_len, batch) format
        x = x.transpose(0, 1)

        # Forward through RetNet
        out, _ = self.retnet(
            prev_output_tokens=x.transpose(0, 1),
            features_only=True
        )

        # Output logits
        return self.fc_out(out)


# ── Initialize RetNet for All Seq Lengths ─────────────────────
retnet_models = {}

for seq_len in SEQ_LENGTHS:
    model = RetNetModel(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        seq_len=seq_len,
        dropout=DROPOUT
    ).to(device)

    retnet_models[seq_len] = model
    print(f"✅ RetNet seq_len={seq_len} → "
          f"Parameters: {count_params(model):,}")


# ── Sanity Check ──────────────────────────────────────────────
print("\nRunning RetNet forward pass check...")
model = retnet_models[128]
model.eval()

x, y = next(iter(dataloaders[128]['train']))
x    = x.to(device)

with torch.no_grad():
    output = model(x)

print(f"  Input  : {x.shape}")
print(f"  Output : {output.shape}")
print(f"✅ RetNet forward pass works!")

✅ RetNet seq_len=128 → Parameters: 21,512,529

Running RetNet forward pass check...
  Input  : torch.Size([8, 128])
  Output : torch.Size([8, 128, 50257])
✅ RetNet forward pass works!


In [24]:
# ── Final Parameter Comparison ────────────────────────────────
print("\n── Parameter Count Comparison ──")
print(f"  Transformer : {count_params(transformer_models[128]):,}")
print(f"  RWKV        : {count_params(rwkv_models[128]):,}")
print(f"  RetNet      : {count_params(retnet_models[128]):,}")
print("\n✅ All 3 models ready!")


── Parameter Count Comparison ──
  Transformer : 13,726,161
  RWKV        : 13,559,505
  RetNet      : 21,512,529

✅ All 3 models ready!


In [25]:
def forward(self, x):
    B, T = x.shape

    # Try this format if above fails
    out, _ = self.retnet(
        prev_output_tokens=x,
        features_only=True,
        return_all_hiddens=False
    )
    return self.fc_out(out)

In [26]:
# ── Metrics Helper Functions ──────────────────────────────────

def get_gpu_utilization():
    """Returns GPU utilization percentage"""
    if GPU_AVAILABLE:
        res = pynvml.nvmlDeviceGetUtilizationRates(handle)
        return res.gpu
    return 0.0

def get_vram_used():
    """Returns peak VRAM used in GB"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1e9
    return 0.0

def compute_perplexity(loss):
    """Perplexity = exp(loss)"""
    return math.exp(loss)


# ── Training Loop ─────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    torch.cuda.reset_peak_memory_stats()

    total_loss    = 0
    total_tokens  = 0
    gpu_utils     = []
    start_time    = time.time()

    for batch_idx, (x, y) in enumerate(loader):
        x = x.to(device)
        y = y.to(device)

        # Forward pass
        optimizer.zero_grad()
        output = model(x)

        # Compute loss
        # output: [B, T, vocab_size] → [B*T, vocab_size]
        # y:      [B, T]             → [B*T]
        loss = criterion(
            output.reshape(-1, output.size(-1)),
            y.reshape(-1)
        )

        # Backward pass
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        # Accumulate metrics
        total_loss   += loss.item()
        total_tokens += x.numel()
        gpu_utils.append(get_gpu_utilization())

    # Compute epoch metrics
    epoch_time  = time.time() - start_time
    avg_loss    = total_loss / len(loader)
    tokens_sec  = total_tokens / epoch_time
    vram        = get_vram_used()
    gpu_util    = sum(gpu_utils) / len(gpu_utils)

    return {
        'loss'       : avg_loss,
        'tokens_sec' : tokens_sec,
        'epoch_time' : epoch_time,
        'vram'       : vram,
        'gpu_util'   : gpu_util
    }


# ── Validation Loop ───────────────────────────────────────────

def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss = criterion(
                output.reshape(-1, output.size(-1)),
                y.reshape(-1)
            )

            total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    ppl      = compute_perplexity(avg_loss)

    return {
        'loss' : avg_loss,
        'ppl'  : ppl
    }


# ── Full Training Function ────────────────────────────────────

def train_model(model_name, model, train_loader,
                val_loader, epochs, lr, device):

    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")

    # Optimizer and loss
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    # History tracking
    history = {
        'train_loss'  : [],
        'val_loss'    : [],
        'val_ppl'     : [],
        'tokens_sec'  : [],
        'epoch_time'  : [],
        'vram'        : [],
        'gpu_util'    : []
    }

    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        # Train
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        # Validate
        val_metrics = validate(
            model, val_loader, criterion, device
        )

        # Save history
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_ppl'].append(val_metrics['ppl'])
        history['tokens_sec'].append(train_metrics['tokens_sec'])
        history['epoch_time'].append(train_metrics['epoch_time'])
        history['vram'].append(train_metrics['vram'])
        history['gpu_util'].append(train_metrics['gpu_util'])

        # Print progress
        print(f"  Train Loss  : {train_metrics['loss']:.4f}")
        print(f"  Val Loss    : {val_metrics['loss']:.4f}")
        print(f"  Perplexity  : {val_metrics['ppl']:.2f}")
        print(f"  Tokens/sec  : {train_metrics['tokens_sec']:,.0f}")
        print(f"  Epoch Time  : {train_metrics['epoch_time']:.1f}s")
        print(f"  Peak VRAM   : {train_metrics['vram']:.2f} GB")
        print(f"  GPU Util    : {train_metrics['gpu_util']:.1f}%")

        # Save best model
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            torch.save(model.state_dict(), f"{model_name}_best.pt")
            print(f"  ✅ Best model saved")

    print(f"\n✅ {model_name} training complete!")
    print(f"   Best Val Loss : {best_val_loss:.4f}")
    print(f"   Best PPL      : {compute_perplexity(best_val_loss):.2f}")

    return history


print("✅ Training pipeline ready!")

✅ Training pipeline ready!


In [27]:
# ── Quick Pipeline Test ───────────────────────────────────────
print("Testing pipeline with 2 batches...")

# Grab just 2 batches
from itertools import islice

mini_train = list(islice(dataloaders[128]['train'], 2))
mini_val   = list(islice(dataloaders[128]['val'],   2))

# Wrap as loader
mini_train_loader = [(x, y) for x, y in mini_train]
mini_val_loader   = [(x, y) for x, y in mini_val]

# Test on Transformer
test_model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    seq_len=128,
    dropout=DROPOUT
).to(device)

optimizer = optim.AdamW(test_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Single forward + backward
x, y = mini_train_loader[0]
x, y = x.to(device), y.to(device)

output = test_model(x)
loss   = criterion(
    output.reshape(-1, output.size(-1)),
    y.reshape(-1)
)
loss.backward()

print(f"  Loss : {loss.item():.4f}")
print(f"  ✅ Pipeline test passed!")

# Cleanup
del test_model
torch.cuda.empty_cache()

Testing pipeline with 2 batches...
  Loss : 10.8306
  ✅ Pipeline test passed!


In [28]:
# ── Safe Experiment Runner ────────────────────────────────────
import gc

# ── JSON Serialization Helper ────────────────────────────────
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (int, float, str, bool)) or obj is None:
        return obj
    elif hasattr(obj, 'item'):
        return obj.item()
    else:
        return float(obj)


all_results = {}

for model_name in ['Transformer', 'RWKV', 'RetNet']:
    all_results[model_name] = {}

    for seq_len in SEQ_LENGTHS:
        print(f"\n{'#'*60}")
        print(f"# {model_name} — seq_len={seq_len}")
        print(f"{'#'*60}")

        # ── Build fresh model ─────────────────────────────────
        torch.cuda.empty_cache()
        gc.collect()

        if model_name == 'Transformer':
            model = TransformerModel(
                vocab_size=VOCAB_SIZE,
                embed_dim=EMBED_DIM,
                num_heads=NUM_HEADS,
                num_layers=NUM_LAYERS,
                seq_len=seq_len,
                dropout=DROPOUT
            ).to(device)

        elif model_name == 'RWKV':
            model = RWKVModel(
                vocab_size=VOCAB_SIZE,
                embed_dim=EMBED_DIM,
                num_layers=NUM_LAYERS,
                seq_len=seq_len,
                dropout=DROPOUT
            ).to(device)

        elif model_name == 'RetNet':
            model = RetNetModel(
                vocab_size=VOCAB_SIZE,
                embed_dim=EMBED_DIM,
                num_heads=NUM_HEADS,
                num_layers=NUM_LAYERS,
                seq_len=seq_len,
                dropout=DROPOUT
            ).to(device)

        print(f"Parameters: {count_params(model):,}")
        print(f"VRAM before training: "
              f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

        # ── Train ─────────────────────────────────────────────
        history = train_model(
            model_name   = f"{model_name}_seq{seq_len}",
            model        = model,
            train_loader = dataloaders[seq_len]['train'],
            val_loader   = dataloaders[seq_len]['val'],
            epochs       = EPOCHS,
            lr           = LR,
            device       = device
        )

        # ── Store results ─────────────────────────────────────
        all_results[model_name][seq_len] = {
            'history'          : history,
            'final_train_loss' : history['train_loss'][-1],
            'final_val_loss'   : history['val_loss'][-1],
            'final_ppl'        : history['val_ppl'][-1],
            'avg_tokens_sec'   : sum(history['tokens_sec']) / len(history['tokens_sec']),
            'avg_epoch_time'   : sum(history['epoch_time']) / len(history['epoch_time']),
            'peak_vram'        : max(history['vram']),
            'avg_gpu_util'     : sum(history['gpu_util']) / len(history['gpu_util'])
        }

        # ── Save immediately after each run ───────────────────
        with open('benchmark_results.json', 'w') as f:
            json.dump(make_serializable(all_results), f, indent=2)
        print(f"✅ Results saved to JSON")

        # ── Delete model and free memory ──────────────────────
        del model
        torch.cuda.empty_cache()
        gc.collect()

        print(f"VRAM after cleanup: "
              f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"\n{'='*60}")
print("✅ ALL EXPERIMENTS COMPLETE!")
print(f"{'='*60}")


############################################################
# Transformer — seq_len=128
############################################################
Parameters: 13,726,161
VRAM before training: 0.58 GB

Training Transformer_seq128

Epoch 1/5
  Train Loss  : 5.0422
  Val Loss    : 3.7982
  Perplexity  : 44.62
  Tokens/sec  : 32,344
  Epoch Time  : 35.6s
  Peak VRAM   : 1.56 GB
  GPU Util    : 82.6%
  ✅ Best model saved

Epoch 2/5
  Train Loss  : 3.6500
  Val Loss    : 3.3339
  Perplexity  : 28.05
  Tokens/sec  : 29,316
  Epoch Time  : 39.3s
  Peak VRAM   : 1.56 GB
  GPU Util    : 80.8%
  ✅ Best model saved

Epoch 3/5
  Train Loss  : 3.0195
  Val Loss    : 1.6197
  Perplexity  : 5.05
  Tokens/sec  : 31,737
  Epoch Time  : 36.3s
  Peak VRAM   : 1.56 GB
  GPU Util    : 83.2%
  ✅ Best model saved

Epoch 4/5
  Train Loss  : 1.0619
  Val Loss    : 0.4754
  Perplexity  : 1.61
  Tokens/sec  : 31,646
  Epoch Time  : 36.4s
  Peak VRAM   : 1.56 GB
  GPU Util    : 83.8%
  ✅ Best model saved

Epoc

NameError: name 'make_serializable' is not defined

In [ ]:
# ── Save Results to JSON ──────────────────────────────────────
import json

# make_serializable is defined before the experiment runner.
results_clean = make_serializable(all_results)

with open('benchmark_results.json', 'w') as f:
    json.dump(results_clean, f, indent=2)

print("✅ Results saved to benchmark_results.json")


# ── Print Master Summary Table ────────────────────────────────
print(f"\n{'='*75}")
print(f"{'Model':<15} {'SeqLen':<10} {'TrainLoss':<12} {'ValLoss':<12} {'PPL':<10} {'Tok/s':<12} {'VRAM(GB)':<10}")
print(f"{'='*75}")

for model_name in ['Transformer', 'RWKV', 'RetNet']:
    for seq_len in SEQ_LENGTHS:
        r = all_results[model_name][seq_len]
        print(
            f"{model_name:<15} "
            f"{seq_len:<10} "
            f"{r['final_train_loss']:<12.4f} "
            f"{r['final_val_loss']:<12.4f} "
            f"{r['final_ppl']:<10.2f} "
            f"{r['avg_tokens_sec']:<12.0f} "
            f"{r['peak_vram']:<10.2f}"
        )

print(f"{'='*75}")

In [ ]:
# Reload saved results
with open('benchmark_results.json', 'r') as f:
    all_results = json.load(f)

print("✅ Results reloaded from backup")

In [ ]:
# Mount drive and copy results
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    'benchmark_results.json',
    '/content/drive/MyDrive/benchmark_results.json'
)
print("✅ Saved to Google Drive")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Plot Config ───────────────────────────────────────────────
sns.set_style("whitegrid")
COLORS    = {
    'Transformer' : '#2196F3',   # Blue
    'RWKV'        : '#4CAF50',   # Green
    'RetNet'      : '#FF5722'    # Orange
}
MODELS    = ['Transformer', 'RWKV', 'RetNet']
REP_SEQ   = SEQ_LENGTHS[-1]
FIG_SIZE  = (10, 5)

def result_for(model_name, seq_len=REP_SEQ):
    model_results = all_results[model_name]
    return model_results.get(seq_len, model_results.get(str(seq_len)))

# ── Helper ────────────────────────────────────────────────────
def save_fig(name):
    plt.tight_layout()
    plt.savefig(f"{name}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved {name}.png")


# ════════════════════════════════════════════════════════════
# GRAPH 1 — Training Loss vs Epoch
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

for model_name in MODELS:
    # Use the configured representative sequence length
    history     = result_for(model_name)['history']
    train_loss  = history['train_loss']
    epochs      = range(1, len(train_loss) + 1)

    plt.plot(
        epochs, train_loss,
        label=model_name,
        color=COLORS[model_name],
        linewidth=2,
        marker='o',
        markersize=5
    )

plt.title('Training Loss vs Epoch',    fontsize=14, fontweight='bold')
plt.xlabel('Epoch',                    fontsize=12)
plt.ylabel('Training Loss',            fontsize=12)
plt.legend(fontsize=11)
plt.xticks(range(1, EPOCHS + 1))
save_fig('graph1_train_loss')


# ════════════════════════════════════════════════════════════
# GRAPH 2 — Validation Loss vs Epoch
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

for model_name in MODELS:
    history   = result_for(model_name)['history']
    val_loss  = history['val_loss']
    epochs    = range(1, len(val_loss) + 1)

    plt.plot(
        epochs, val_loss,
        label=model_name,
        color=COLORS[model_name],
        linewidth=2,
        marker='s',
        markersize=5
    )

plt.title('Validation Loss vs Epoch',  fontsize=14, fontweight='bold')
plt.xlabel('Epoch',                    fontsize=12)
plt.ylabel('Validation Loss',          fontsize=12)
plt.legend(fontsize=11)
plt.xticks(range(1, EPOCHS + 1))
save_fig('graph2_val_loss')


# ════════════════════════════════════════════════════════════
# GRAPH 3 — Perplexity Comparison (Bar Chart)
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

ppls  = [result_for(m)['final_ppl'] for m in MODELS]
bars  = plt.bar(
    MODELS, ppls,
    color=[COLORS[m] for m in MODELS],
    width=0.5,
    edgecolor='white',
    linewidth=1.5
)

# Add value labels on bars
for bar, val in zip(bars, ppls):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f}',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title(f'Perplexity Comparison (seq_len={REP_SEQ})',   fontsize=14, fontweight='bold')
plt.xlabel('Model',                                fontsize=12)
plt.ylabel('Perplexity (lower = better)',          fontsize=12)
save_fig('graph3_perplexity')


# ════════════════════════════════════════════════════════════
# GRAPH 4 — Tokens/sec Comparison (Bar Chart)
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

toks  = [result_for(m)['avg_tokens_sec'] for m in MODELS]
bars  = plt.bar(
    MODELS, toks,
    color=[COLORS[m] for m in MODELS],
    width=0.5,
    edgecolor='white',
    linewidth=1.5
)

for bar, val in zip(bars, toks):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f'{val:,.0f}',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title(f'Throughput Comparison (seq_len={REP_SEQ})',   fontsize=14, fontweight='bold')
plt.xlabel('Model',                                fontsize=12)
plt.ylabel('Tokens/sec (higher = better)',         fontsize=12)
save_fig('graph4_tokens_sec')


# ════════════════════════════════════════════════════════════
# GRAPH 5 — Peak VRAM Comparison (Bar Chart)
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

vrams = [result_for(m)['peak_vram'] for m in MODELS]
bars  = plt.bar(
    MODELS, vrams,
    color=[COLORS[m] for m in MODELS],
    width=0.5,
    edgecolor='white',
    linewidth=1.5
)

for bar, val in zip(bars, vrams):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{val:.2f} GB',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title(f'Peak VRAM Comparison (seq_len={REP_SEQ})',    fontsize=14, fontweight='bold')
plt.xlabel('Model',                                fontsize=12)
plt.ylabel('Peak VRAM in GB (lower = better)',     fontsize=12)
save_fig('graph5_vram')


# ════════════════════════════════════════════════════════════
# GRAPH 6 — GPU Utilization Comparison (Bar Chart)
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

gpu_utils = [result_for(m)['avg_gpu_util'] for m in MODELS]
bars      = plt.bar(
    MODELS, gpu_utils,
    color=[COLORS[m] for m in MODELS],
    width=0.5,
    edgecolor='white',
    linewidth=1.5
)

for bar, val in zip(bars, gpu_utils):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title(f'GPU Utilization Comparison (seq_len={REP_SEQ})',  fontsize=14, fontweight='bold')
plt.xlabel('Model',                                    fontsize=12)
plt.ylabel('GPU Utilization % (higher = better used)', fontsize=12)
plt.ylim(0, 110)
save_fig('graph6_gpu_util')


# ════════════════════════════════════════════════════════════
# GRAPH 7 — Perplexity vs Context Length
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

for model_name in MODELS:
    ppls = [
        result_for(model_name, seq)['final_ppl']
        for seq in SEQ_LENGTHS
    ]
    plt.plot(
        SEQ_LENGTHS, ppls,
        label=model_name,
        color=COLORS[model_name],
        linewidth=2,
        marker='o',
        markersize=8
    )

plt.title('Perplexity vs Context Length',  fontsize=14, fontweight='bold')
plt.xlabel('Context Length',               fontsize=12)
plt.ylabel('Perplexity (lower = better)',  fontsize=12)
plt.xticks(SEQ_LENGTHS)
plt.legend(fontsize=11)
save_fig('graph7_ppl_vs_context')


# ════════════════════════════════════════════════════════════
# GRAPH 8 — Throughput vs Context Length
# ════════════════════════════════════════════════════════════
plt.figure(figsize=FIG_SIZE)

for model_name in MODELS:
    toks = [
        result_for(model_name, seq)['avg_tokens_sec']
        for seq in SEQ_LENGTHS
    ]
    plt.plot(
        SEQ_LENGTHS, toks,
        label=model_name,
        color=COLORS[model_name],
        linewidth=2,
        marker='o',
        markersize=8
    )

plt.title('Throughput vs Context Length',      fontsize=14, fontweight='bold')
plt.xlabel('Context Length',                   fontsize=12)
plt.ylabel('Tokens/sec (higher = better)',     fontsize=12)
plt.xticks(SEQ_LENGTHS)
plt.legend(fontsize=11)
save_fig('graph8_throughput_vs_context')


# ════════════════════════════════════════════════════════════
# BONUS — All 8 Graphs in One Figure
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle(
    'Benchmark Results: Transformer vs RWKV vs RetNet',
    fontsize=16, fontweight='bold', y=1.02
)

# Graph 1 — Train Loss
ax = axes[0][0]
for model_name in MODELS:
    history = result_for(model_name)['history']
    ax.plot(
        range(1, EPOCHS+1),
        history['train_loss'],
        label=model_name,
        color=COLORS[model_name],
        linewidth=2, marker='o'
    )
ax.set_title('Training Loss vs Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# Graph 2 — Val Loss
ax = axes[0][1]
for model_name in MODELS:
    history = result_for(model_name)['history']
    ax.plot(
        range(1, EPOCHS+1),
        history['val_loss'],
        label=model_name,
        color=COLORS[model_name],
        linewidth=2, marker='s'
    )
ax.set_title('Validation Loss vs Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# Graph 3 — Perplexity Bar
ax = axes[0][2]
ppls = [result_for(m)['final_ppl'] for m in MODELS]
ax.bar(MODELS, ppls, color=[COLORS[m] for m in MODELS], width=0.5)
ax.set_title('Perplexity Comparison')
ax.set_ylabel('Perplexity')

# Graph 4 — Tokens/sec Bar
ax = axes[0][3]
toks = [result_for(m)['avg_tokens_sec'] for m in MODELS]
ax.bar(MODELS, toks, color=[COLORS[m] for m in MODELS], width=0.5)
ax.set_title('Throughput Comparison')
ax.set_ylabel('Tokens/sec')

# Graph 5 — VRAM Bar
ax = axes[1][0]
vrams = [result_for(m)['peak_vram'] for m in MODELS]
ax.bar(MODELS, vrams, color=[COLORS[m] for m in MODELS], width=0.5)
ax.set_title('Peak VRAM Comparison')
ax.set_ylabel('GB')

# Graph 6 — GPU Util Bar
ax = axes[1][1]
utils = [result_for(m)['avg_gpu_util'] for m in MODELS]
ax.bar(MODELS, utils, color=[COLORS[m] for m in MODELS], width=0.5)
ax.set_title('GPU Utilization')
ax.set_ylabel('%')

# Graph 7 — PPL vs Context
ax = axes[1][2]
for model_name in MODELS:
    ppls = [result_for(model_name, s)['final_ppl'] for s in SEQ_LENGTHS]
    ax.plot(SEQ_LENGTHS, ppls, label=model_name,
            color=COLORS[model_name], linewidth=2, marker='o')
ax.set_title('Perplexity vs Context Length')
ax.set_xlabel('Context Length')
ax.set_ylabel('Perplexity')
ax.legend()

# Graph 8 — Throughput vs Context
ax = axes[1][3]
for model_name in MODELS:
    toks = [result_for(model_name, s)['avg_tokens_sec'] for s in SEQ_LENGTHS]
    ax.plot(SEQ_LENGTHS, toks, label=model_name,
            color=COLORS[model_name], linewidth=2, marker='o')
ax.set_title('Throughput vs Context Length')
ax.set_xlabel('Context Length')
ax.set_ylabel('Tokens/sec')
ax.legend()

plt.tight_layout()
plt.savefig('all_graphs.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ All 8 graphs saved!")

In [ ]:
# ── Save All Graphs to Drive ──────────────────────────────────
graph_files = [
    'graph1_train_loss.png',
    'graph2_val_loss.png',
    'graph3_perplexity.png',
    'graph4_tokens_sec.png',
    'graph5_vram.png',
    'graph6_gpu_util.png',
    'graph7_ppl_vs_context.png',
    'graph8_throughput_vs_context.png',
    'all_graphs.png'
]

for f in graph_files:
    shutil.copy(f, f'/content/drive/MyDrive/{f}')

print("✅ All graphs saved to Google Drive!")